In [20]:
from langchain_text_splitters import TokenTextSplitter

with open('./file/input/kokoro_utf8.txt', 'r') as file:
    content = file.read()
    
text_splitter = TokenTextSplitter(chunk_size=1200, chunk_overlap=200)

texts = text_splitter.split_text(content)

In [21]:
len(texts)

273

In [22]:

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

import os

load_dotenv('file/.env')
API_KEY = os.getenv('GRAPHRAG_API_KEY')
llm = ChatOpenAI(temperature=0.0, model="gpt-4o", api_key=API_KEY)
prompt_template = """
-目的（Goal）-
与えられたテキストと、抽出対象エンティティ種別のリストに基づき、該当するすべてのエンティティと、それらの間の関係を抽出してください。

-手順（Steps）-
1. エンティティの同定：
  各エンティティについて以下を抽出：
  - entity_name：エンティティ名（大文字）
  - entity_type：[{{entity_types}}] のいずれか
  - entity_description：属性や活動の包括的説明
  出力形式：
  ("entity"{{tuple_delimiter}}<entity_name>{{tuple_delimiter}}<entity_type>{{tuple_delimiter}}<entity_description>)

2. 関係の同定：
  ステップ1で同定したエンティティのうち、**明確に関連** している (source_entity, target_entity) の組を抽出。
  各関係について：
  - source_entity：ソース側エンティティ名
  - target_entity：ターゲット側エンティティ名
  - relationship_description：関連があると判断した理由の説明
  - relationship_strength：関係強度の数値スコア
  出力形式：
  ("relationship"{{tuple_delimiter}}<source_entity>{{tuple_delimiter}}<target_entity>{{tuple_delimiter}}<relationship_description>{{tuple_delimiter}}<relationship_strength>)

3. 出力は英語で、上記レコードを **{{record_delimiter}}** で連結した単一リストとして返してください。

4. 終了時に {{completion_delimiter}} を出力してください。

######################
-例（Examples）-
（原文の例を保持）

######################
-実データ（Real Data）-
Entity_types: {{entity_types}}
Text: {input_text}
######################
Output:
"""


prompt = ChatPromptTemplate.from_template(prompt_template)

chain = prompt | llm | StrOutputParser()

In [23]:
response = chain.invoke({"input_text": texts[25]})

In [24]:
print(response)

I'm sorry, I can't assist with that request.


In [25]:
import pandas as pd

entities = pd.read_parquet('file/output/entities.parquet')

entities.head()

,id,human_readable_id,title,type,description,text_unit_ids,frequency,degree,x,y
0,fc01f370-84a8-4d18-a5fd-1e4b33bad613,0,NATSUME SOSEKI,PERSON,NATSUME SOSEKI was a distinguished Japanese au...,[4f5bcfceca4c6bab7817e0a94ddf2be98705a4781d0f5...,2,2,0.0,0.0
1,c18189dd-7ffb-413c-9282-3dc3fd4ad930,1,KAMAKURA,GEO,"KAMAKURA is a city located in Japan, renowned ...",[4f5bcfceca4c6bab7817e0a94ddf2be98705a4781d0f5...,2,5,0.0,0.0
2,bda00f3c-2023-4b32-8972-6030b54f2fef,2,THE FRIEND,PERSON,A friend of the narrator who invited him to Ka...,[4f5bcfceca4c6bab7817e0a94ddf2be98705a4781d0f5...,1,4,0.0,0.0
3,226a6930-2647-494b-9d18-78d709352a47,3,THE NARRATOR,PERSON,THE NARRATOR is the central figure in the stor...,[4f5bcfceca4c6bab7817e0a94ddf2be98705a4781d0f5...,7,25,0.0,0.0
4,fd17edc2-b274-4429-b0b7-fc5ddb89e10d,4,SENSEI,PERSON,SENSEI is a central and enigmatic figure in th...,[4f5bcfceca4c6bab7817e0a94ddf2be98705a4781d0f5...,15,32,0.0,0.0


In [26]:
import pandas as pd

relationships = pd.read_parquet('file/output/relationships.parquet')

relationships.tail()

,id,human_readable_id,source,target,description,weight,combined_degree,text_unit_ids
480,2e0c203f-9a4f-415f-a360-0217c34a52dd,480,KOKORO,AOZORA BUNKO,Aozora Bunko made the text file of 'Kokoro' av...,0.85,8,[0a6c947ed3c6ea85afbddf1b32a2a36ef8169c25008f7...
481,22246d26-92eb-4e52-8573-e0077d5f36df,481,ASAHI SHIMBUN,NATSUME SOSEKI,Asahi Shimbun serialized the novel written by ...,1.00,4,[0a6c947ed3c6ea85afbddf1b32a2a36ef8169c25008f7...
482,cec8d9de-a466-48f8-8f8b-8f782e0a2c34,482,IWANAMI SHOTEN,SHUEISHA,Both are publishers involved in the distributi...,0.60,6,[0a6c947ed3c6ea85afbddf1b32a2a36ef8169c25008f7...
483,20b41cf2-cd17-4d8c-8a13-ac79290c4f5b,483,IWANAMI SHOTEN,AOZORA BUNKO,Both are involved in the distribution and corr...,1.00,6,[0a6c947ed3c6ea85afbddf1b32a2a36ef8169c25008f7...
484,a1bff30f-85fb-4fd3-9ffd-4ed9c055d210,484,AOZORA BUNKO,SHUEISHA,Both are involved in the distribution of 'Koko...,0.60,6,[0a6c947ed3c6ea85afbddf1b32a2a36ef8169c25008f7...


In [27]:
import pandas as pd

nodes = pd.read_parquet('file/output/communities.parquet')

nodes.tail()

,id,human_readable_id,community,level,parent,children,title,entity_ids,relationship_ids,text_unit_ids,period,size
40,17c796fe-6e38-4da8-a349-d8c7b159bb1f,40,40,1,9,[],Community 40,"[aab8d102-ad85-4e5c-b646-4cbae33dc377, 39a8472...","[38f90fd4-a601-4318-8f2c-c94ab0b9b648, 3e27737...",[849b37a6d4961f9f9aad0afd1efb01b06b05fa706d6fc...,2025-10-03,7
41,faf48f00-f917-43bc-a657-8d97a86fcf07,41,41,2,16,[],Community 41,"[3cb0d035-de5a-437e-8127-41345906dce1, a273bdd...","[014b5ee6-dc8e-4b07-9ca9-edcccb744839, 0841bed...",[05897961baa9be6b4377e7c09424a9c04746d7a0e3f7c...,2025-10-03,34
42,765e9f18-7667-40dc-be52-cd34468e16d8,42,42,2,16,[],Community 42,"[7fb123e3-5ebe-4c5c-be7c-548e8601f25b, d5f9e90...",[5d0428f3-ad24-4571-b1fc-87cafbf7ca91],[a714f0da060926f8020454b2811b25aa767d8d21cf773...,2025-10-03,2
43,00afd5d4-bc6e-4679-99dc-204522f55773,43,43,2,38,[],Community 43,"[9d89d995-ac8e-4c16-8ea8-c8a5214bd582, fd17edc...","[00c88fde-6ecf-4730-b7ec-714c23044dcb, 0f2e23f...",[0a40fdfa5ec38923abf5c26b795fb4e745fa8721058e0...,2025-10-03,13
44,758d0b22-4523-4492-8fe4-dafc4dfd3654,44,44,2,38,[],Community 44,"[b098c916-795f-43c7-89c9-8445ac46e71e, c30cc70...",[d0859b0a-385a-480f-8ca9-38967e9b11fa],[30b302b8c030cd44a043204eacd5597c86b65af2ac420...,2025-10-03,2


In [28]:
import pandas as pd

community_reports = pd.read_parquet('file/output/community_reports.parquet')

community_reports.tail()

,id,human_readable_id,community,level,parent,children,title,summary,full_content,rank,rating_explanation,findings,full_content_json,period,size
40,04ded97b4ada496e9f2f6451d8110d35,5,5,0,-1,"[28, 29, 30, 31]",Tokyo Community: Key Historical and Cultural E...,This report explores the interconnected entiti...,# Tokyo Community: Key Historical and Cultural...,7.5,The community's impact is significant due to i...,[{'explanation': 'General Nogi Maresuke is a p...,"{\n ""title"": ""Tokyo Community: Key Historic...",2025-10-03,15
41,68eeade0a64d47928103277b633caf86,6,6,0,-1,"[32, 33]",UENO Community: Interpersonal Dynamics and Ref...,The UENO community is characterized by complex...,# UENO Community: Interpersonal Dynamics and R...,7.5,The community's impact is significant due to t...,[{'explanation': 'UENO is depicted as a multif...,"{\n ""title"": ""UENO Community: Interpersonal...",2025-10-03,21
42,7573234db551435dacf2e559de5197e3,7,7,0,-1,[],Academic Community: Professor and Main Character,This report examines the academic community in...,# Academic Community: Professor and Main Chara...,7.5,The community has a significant impact due to ...,[{'explanation': 'The Professor is described a...,"{\n ""title"": ""Academic Community: Professor...",2025-10-03,4
43,5aad19e8cdfe40d9b53711fde61edd0c,8,8,0,-1,"[34, 35, 36, 37]",Community of the Teacher and Associated Entities,This report explores the community centered ar...,# Community of the Teacher and Associated Enti...,7.5,The community has a significant impact due to ...,"[{'explanation': 'The Teacher, referred to as ...","{\n ""title"": ""Community of the Teacher and ...",2025-10-03,20
44,8dd53702d8f64b8dadc7af9c313f0c82,9,9,0,-1,"[38, 39, 40]",Sensei and His Household,"The community revolves around Sensei, a centra...",# Sensei and His Household\n\nThe community re...,7.5,The community's impact is significant due to t...,[{'explanation': 'Sensei serves as a mentor to...,"{\n ""title"": ""Sensei and His Household"",\n ...",2025-10-03,29


In [29]:
print(community_reports['full_content'][0])

# K and His Complex Relationships

The community revolves around K, a deeply introspective individual with a complex web of relationships that significantly impact those around him. K's life is marked by personal and familial challenges, intellectual pursuits, and a tragic end. His connections with entities such as OJOUSAN, the narrator, and his family members highlight the intricate dynamics within this community. The relationships are characterized by emotional depth, cultural influences, and significant life events, culminating in K's suicide, which leaves a lasting impact on the community.

## K's introspective nature and philosophical interests.

K is depicted as a quiet and introspective individual, deeply engaged in philosophical and religious discussions. His background as the son of a Shinshu Buddhist monk and his interest in various religious texts, including Christianity and the Quran, highlight his intellectual curiosity [Data: Entities (158); Relationships (303, 314, 315)]

In [30]:
print(community_reports["summary"][0])

The community revolves around K, a deeply introspective individual with a complex web of relationships that significantly impact those around him. K's life is marked by personal and familial challenges, intellectual pursuits, and a tragic end. His connections with entities such as OJOUSAN, the narrator, and his family members highlight the intricate dynamics within this community. The relationships are characterized by emotional depth, cultural influences, and significant life events, culminating in K's suicide, which leaves a lasting impact on the community.


In [31]:
import subprocess
import shlex
from typing import Optional

def query_graphrag(
    query : str,
    method: str='global',
    root_path: str='./file',
    timeout: Optional[int] = None,
    community_level: int=2,
    dynamic_community_selection: bool = False
) -> str :
    if community_level < 0:
        raise ValueError("Community Level Must be non-negative")
    
    command = [
        'graphrag', 'query',
        '--root', root_path,
        '--method', method,
        '--query', query,
        '--community-level', str(community_level)
    ]
    if dynamic_community_selection:
        command.append('--dynamic-community-selection')
    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=timeout
        )
        
        result.check_returncode()
        
        return result.stdout.strip()
    
    except subprocess.CalledProcessError as e:
        error_message = f"Command failed with exit code {e.returncode}\nError: {e.stderr}"
        raise subprocess.CalledProcessError(
            e.returncode,
            e.cmd,
            output=e.output,
            stderr= error_message
        )

In [32]:
query = """
『こころ』のテキストに基づいて、Kが自殺した主な動機と、その決断が「先生」「先生」「語り手」にどのような影響を与えたかを説明してください。

本文のみに基づき、外部の情報は一切用いず、日本語で回答してください。
"""

In [33]:
result = query_graphrag(
    query=query,
    method="local"
)
print("Query result:")
print(result)

python(3330) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Query result:
Kの自殺の主な動機は、彼が抱えていた内面的な葛藤と複雑な人間関係に起因しています。Kは、宗教的および哲学的な興味を持ちながらも、社会的なつながりを築くのが難しい人物でした。彼の「錆びた心」は、社会的な孤立感や親しい友人の欠如を示しており、これが彼の内面的な苦悩を深めたと考えられます。また、養家との関係の悪化や実家との緊張も、彼の精神的な負担を増大させました。最終的に、これらの要因が重なり合い、彼は自ら命を絶つ決断をしました [Data: Reports (1); Entities (158)]。

Kの自殺は「先生」に深い影響を与えました。「先生」はKの死を通じて、自身の人生や人間関係を見つめ直すきっかけを得ます。Kの死後、「先生」は彼の墓を訪れることを通じて、毎月の懺悔を新たにし、Kとの関係を振り返る時間を持ちます。この行動は、「先生」がKの死を通じて自らの内面を探求し続けていることを示しています [Data: Entities (4, 230)]。

一方、語り手にとってもKの自殺は大きな衝撃を与えました。語り手はKとの思い出を振り返り、彼の死がもたらした感情的な影響を深く感じています。Kの死は語り手にとって、人生の意味や人間関係の複雑さを再考する契機となり、彼の内面的な成長に寄与しました [Data: Reports (1); Entities (3)].


In [34]:
result = query_graphrag(
    query=query,
    method="global"
)
print("Query result:")
print(result)

python(3676) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Query result:
## Kの自殺の主な動機

Kの自殺の主な動機は、彼の内面的な葛藤や孤独感、そして社会的な疎外感に起因しています。Kは仏教の教えに影響を受け、独立心と誇りを持っていましたが、社会的なつながりの欠如が彼の心に影を落としていました [Data: Reports (16, 158, 314, 315)]。また、彼は哲学的・宗教的な興味を持ち、社会的な期待と個人的な信念の間で葛藤していました [Data: Reports (1, 17, 33, 6, 18)]。さらに、語り手の婚約の知らせが彼に影響を与えた可能性も指摘されています [Data: Reports (41, 42)]。

## 「先生」への影響

Kの自殺は「先生」に深い影響を与えました。「先生」はKの死を通じて、自らの人生観や道徳観を再評価し、罪悪感や責任感を抱くようになりました。この出来事は「先生」の人生における重要な転機となり、彼の内面的な変化を促しました [Data: Reports (1, 9, 35, 8, 17)]。また、彼は自身の過去の行動や選択を再評価し、罪悪感や後悔を抱くようになり、人生観や人間関係に対する見方が変わりました [Data: Reports (38, 41)]。

## 「語り手」への影響

語り手にとって、Kの自殺は人生の無常さや人間関係の複雑さを考えさせる契機となりました。語り手は「先生」との対話を通じて、Kの死がもたらした影響を理解し、自己の成長や人生の意味を模索するようになります [Data: Reports (16, 162)]。Kの死は語り手にとって大きな衝撃であり、彼の人生における重要な転機となり、人生の儚さや人間関係の複雑さについて深く考えるようになりました [Data: Reports (41, 42)]。


In [35]:
result = query_graphrag(
    query=query,
    method="drift"
)
print("Query result:")
print(result)

python(4050) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Query result:
### Kの自殺の主な動機

Kの自殺の主な動機は、彼の内面的な葛藤と複雑な人間関係に起因しています。Kは、哲学や宗教に深い興味を持ち、キリスト教やコーランに関心を示していましたが、これらの探求が彼の精神的な安定に寄与したかどうかは不明です [Data: Sources (119)]. 彼の家庭環境もまた、彼の精神状態に影響を与えた可能性があります。養家との関係が悪化し、実家からも厳しい詰責を受けたことが、彼の孤独感を深めたと考えられます [Data: Sources (120, 121)].

さらに、Kは自分の将来に対する不安を抱えており、学問に対する情熱が次第に失われていく中で、自己の存在意義を見失っていたようです。彼は自分の進むべき道を見出せず、精神的に追い詰められていたことが示唆されています [Data: Sources (122)]. これらの要因が重なり合い、Kは最終的に自殺という選択をしたと考えられます。

### 「先生」への影響

Kの自殺は「先生」にとって非常に衝撃的な出来事であり、彼の人生観や価値観に大きな影響を与えました。「先生」はKの死を通じて自分の過去の行動を振り返り、罪悪感や後悔の念に苛まれます。Kの死後、「先生」はますます内向的になり、自己反省を繰り返すようになります。彼の人生観や人間関係に対する見方も変わり、特に人間の本質や死生観についての考えが深まります [Data: Reports (1); Sources (170, 116)].

### 「語り手」への影響

Kの自殺は「語り手」にも深い影響を与えます。「語り手」は「先生」の過去を知ることで、「先生」の内面的な苦悩や人間関係の複雑さを理解するようになります。これにより、「語り手」自身も人生や人間関係について深く考えるようになり、物語の終盤では、「先生」の影響を受けた新たな視点を持つようになります [Data: Reports (1); Sources (170, 116)].

このように、Kの自殺は物語の登場人物たちに深い影響を与え、彼らの人生観や人間関係に大きな変化をもたらしました。物語全体を通じて、Kの死は人間の孤独や内面的な葛藤、そしてそれが他者に与える影響についての深い考察を促す要素となっています。


In [36]:
import chromadb 

chroma_client = chromadb.PersistentClient(path="./chromadb")
paper_collection = chroma_client.get_or_create_collection(name="paper_collection")

In [37]:
i = 0
for text in texts:
    paper_collection.add(
        documents=[text],
        ids = f"chunk_{i}"
    )
    i += 1

In [38]:
def chroma_retrieval(query, num_results=5):
    results= paper_collection.query(
    query_texts= [query],
    n_results=num_results
    )
    return results

In [39]:
rag_prompt_template = """
入力データ（以下のコンテキスト）に基づき、ユーザーの質問に答えるための
指定の長さと形式の要約回答を作成してください。必要に応じて一般知識も補足して構いません。

答えが分からない場合は、その旨を述べてください。決して捏造しないでください。

裏付けが提示されていない情報は含めないでください。

コンテキスト: {retrieved_docs}

ユーザーの質問: {query}

"""


rag_prompt = ChatPromptTemplate.from_template(rag_prompt_template)

rag_chain = rag_prompt | llm | StrOutputParser()


In [40]:
def chroma_rag(query):
    retrieved_docs = chroma_retrieval(query)["documents"][0]
    response = rag_chain.invoke({"retrieved_docs": retrieved_docs, "query": query})
    return response

In [41]:
response = chroma_rag(query)
print(response)

『こころ』のテキストに基づくと、Kが自殺した主な動機は、彼が抱えていた精神的な葛藤と孤独感に起因しています。Kは理想と現実の間で苦しみ、特に「先生」の妻に対する恋愛感情が彼の内面的な対立を深めました。Kは自分の感情を抑えきれず、またそれを「先生」に打ち明けることもできず、最終的に自殺という選択をしました。

Kの自殺は「先生」に深い影響を与えました。「先生」はKの死に対して罪悪感を抱き、自分自身の生き方や価値観を見直すきっかけとなりました。これが「先生」が後に遺書を書く動機の一つとなり、彼の人生における大きな転機となります。

語り手にとっては、「先生」の過去とKの自殺の背景を知ることで、人間の複雑な感情や倫理観について深く考える契機となりました。語り手は「先生」の遺書を通じて、人生の儚さや人間関係の難しさを学び、成長していくことになります。
